<div class="jcs-title"><div class="jcs-kicker">Human–AI Commons · JCS · 02</div>
<h1>Exploring saved worlds</h1>
<p>Read a result bundle, compare designs and follow local resource histories.</p></div>

Averages answer which outcomes were attained. Distributions reveal who gained or lost across worlds. Maps and time histories show how a particular world developed. This notebook connects those views without launching simulations or writing into the bundle.

It supports JCS Pareto result bundles with the saved test-comparison schema. Other compatible bundles can be selected by changing the path below. Summary tables work without detailed trajectories; trajectory controls become available when the corresponding world and history files are present.

**Setup:** complete the [JCS quick start](../../README.md#jcs-quick-start) once. From the repository root, launch `.venv/bin/jupyter lab jcs/notebooks/` and select its **Python 3** kernel. [Model and mathematics](01_model_and_mathematics.ipynb) introduces the equations and small examples. No production run is launched here, including when data are missing.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

%matplotlib inline

here = Path.cwd().resolve()
JCS_ROOT = next((p for p in [here, *here.parents] if (p / "src/human_ai_commons").is_dir()), None)
if JCS_ROOT is None and (here / "jcs/src/human_ai_commons").is_dir():
    JCS_ROOT = here / "jcs"
if JCS_ROOT is None:
    raise RuntimeError("Open the notebook from the JCS folder or its parent repository.")
for path in [JCS_ROOT / "src", JCS_ROOT / "notebooks"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from _explore import style, BLUE, TEAL, ORANGE, GRAY, frame_map, map_legend
style()
display(HTML("""<style>
.jcs-title {border-left:5px solid #278477;padding:12px 22px;background:#f3f7f7;margin:16px 0 24px}
.jcs-title h1 {margin:4px 0 12px;font-weight:600;color:#244867}
.jcs-kicker {font-size:12px;letter-spacing:2px;color:#278477;text-transform:uppercase}
.jcs-note {border-left:3px solid #bc703a;padding:10px 16px;background:#fbf7f2;margin:16px 0}
</style>"""))

from _explore import Bundle
from human_ai_commons.payoffs import Policy
from human_ai_commons.analysis import combined_frontiers


## 1 · Open a bundle and establish its scope

A bundle's manifest identifies its configuration, source version, random-stream convention and completed stages. Those records describe the experiment that produced the data. They need not match a later interface version in the working tree.

The default path is the main experiment's local results folder. Edit `BUNDLE_PATH` for a downloaded bundle or your own compatible experiment. A missing bundle leaves the data views inactive; it does not substitute illustrative data for research results.


In [ ]:
BUNDLE_PATH = JCS_ROOT / "results" / "research_protocol"
bundle = Bundle(BUNDLE_PATH) if (BUNDLE_PATH / "manifest.json").is_file() else None
if bundle is None:
    display(Markdown("**No bundle selected.** Set `BUNDLE_PATH` to a saved JCS result folder, then rerun from this cell. "
                     "The companion model notebook works without research data."))
else:
    cfg = bundle.config
    reference = Policy(t=cfg["payoffs"]["t"], s=cfg["payoffs"]["s"])
    display(pd.DataFrame({"Item": ["Run ID", "World", "Human / AI actors", "Steps / excluded initial steps", "Allocation", "Random streams"],
        "Value": [bundle.run_id, f'{cfg["world"]["width"]} × {cfg["world"]["height"]}',
                  f'{cfg["world"]["humans"]} / {cfg["world"]["ai"]}',
                  f'{cfg["objective"]["total_steps"]} / {cfg["objective"]["excluded_initial_steps"]}',
                  cfg["world"]["withdrawal_allocation"], bundle.manifest["environment"].get("rng", "See manifest")]}))
    display(pd.DataFrame(list(bundle.manifest["stages"].items()), columns=["Recorded stage", "Status"]))


<div class="jcs-note"><strong>Source verification.</strong> The notebook performs local shape and pairing checks. A complete archived-file checksum verification is available separately through <code>.venv/bin/python jcs/reproduce.py verify /path/to/bundle</code> from the repository root. It reads the full archive and can take time. Opening this notebook does not silently run that expensive operation.</div>

## 2 · Which trade-offs were attained?

Each dot is a tested design under the **same pressure, payoff budget and budget norm**. Filled dots lie on the combined three-objective sample frontier: no tested alternative improves one of human support, AI support or depletion without worsening another. Frontier membership uses all three outcomes, even when only two are drawn.

Activity axes show **relative changes in ensemble means**, $100(\overline H/\overline H_0-1)$ and $100(\overline A/\overline A_0-1)$. The subscript zero denotes the original game in the same setting. A visually spread-out frontier can still occupy a tiny numerical range. The absolute-outcome view makes that scale visible. Colour measures the relative depletion change; negative values mean less depletion. When the original score is zero, the colour bar instead shows the absolute score.

Depletion is calculated from the worst source histories; it is not the percentage of the landscape affected. Neither the mean frontier nor the marker colour establishes statistical dominance.


In [ ]:
candidate_table = None
if bundle is not None:
    try:
        candidate_table = bundle.table("pareto_candidates.csv")
        settings = candidate_table[["pressure", "budget", "norm"]].drop_duplicates().sort_values(["pressure", "budget", "norm"])
    except FileNotFoundError:
        display(Markdown("The candidate table is absent. The standard `analyze` command creates it for completed Pareto bundles."))

def show_tradeoffs(setting=0, scale="relative"):
    p, b, norm = settings.iloc[setting]
    rows = candidate_table[(candidate_table.pressure == p) & (candidate_table.budget == b) & (candidate_table.norm == norm)]
    rows = combined_frontiers(rows).copy()
    baseline = rows[rows.policy_id == reference.id].iloc[0]
    if scale == "relative":
        if min(baseline.human, baseline.ai) <= 0:
            display(Markdown("Relative support changes require positive original means. Select absolute coverage for this setting."))
            return
        xx = 100 * (rows.human / baseline.human - 1)
        yy = 100 * (rows.ai / baseline.ai - 1)
        labels = ("Human support change (% of original)", "AI support change (% of original)")
        origin = (0, 0)
    else:
        xx, yy = 100 * rows.human, 100 * rows.ai
        labels = ("Human activity measure (%)", "AI activity measure (%)")
        origin = (100 * baseline.human, 100 * baseline.ai)
    fig, ax = plt.subplots(figsize=(8, 4.1), layout="constrained")
    front = rows.unconstrained_combined_sample_frontier.to_numpy(bool)
    ax.scatter(xx[~front], yy[~front], facecolors="none", edgecolors=GRAY, s=50, label="Other tested designs")
    colour = 100 * (rows.depletion / baseline.depletion - 1) if baseline.depletion > 0 else rows.depletion
    colour_label = "Depletion change (% of original)" if baseline.depletion > 0 else "Concentrated depletion score"
    points = ax.scatter(xx[front], yy[front], c=colour[front], cmap="viridis", s=65, edgecolors=BLUE, linewidths=.5, label="Three-objective mean frontier")
    ax.scatter(*origin, marker="*", s=150, color=ORANGE, edgecolor="white", label="Original", zorder=5)
    if scale == "relative":
        ax.axhline(0, color=GRAY, lw=.5); ax.axvline(0, color=GRAY, lw=.5)
    else:
        ax.set(xlim=(0, 100), ylim=(0, 100))
    ax.set(xlabel=labels[0], ylabel=labels[1], title=f"Pressure {p:g} · budget {b:g} · {norm}")
    ax.ticklabel_format(useOffset=False, style="plain")
    fig.colorbar(points, ax=ax, label=colour_label, format="%.3g")
    ax.legend(frameon=False, fontsize=8, loc="best")
    plt.show(); plt.close(fig)
    display(Markdown(f"**{len(rows)} tested designs; {int(front.sum())} on the unconstrained mean frontier.** "
                     f"{int(rows.constraint_supported.sum())} establish the stated resource condition. "
                     f"Human means span **{100*rows.human.min():.5f}–{100*rows.human.max():.5f}%** on the activity scale; "
                     f"AI means span **{100*rows.ai.min():.5f}–{100*rows.ai.max():.5f}%**. These are means, not confidence limits."))

if candidate_table is not None:
    setting_options = [(f"Pressure {r.pressure:g} · budget {r.budget:g} · {r.norm}", i) for i, r in settings.reset_index(drop=True).iterrows()]
    setting_control = widgets.Dropdown(options=setting_options, description="Setting", layout=widgets.Layout(width="440px"))
    scale_control = widgets.ToggleButtons(options=[("Relative to original", "relative"), ("Absolute activity", "absolute")])
    tradeoff_output = widgets.interactive_output(show_tradeoffs, {"setting": setting_control, "scale": scale_control})
    display(setting_control, scale_control, tradeoff_output)


## 3 · Do the same worlds gain or lose?

A design's mean can conceal gains in some worlds and losses in others. The paired data below compare each design with the original under matching world indices. For the primary linear measure, differences are in **percentage points of the fixed activity target**. If another bundle uses transformed utility or a different value for additional delivery, they are points on that configured activity scale:

$$\Delta H_k=100(H_k-H_{0,k}),\qquad \Delta A_k=100(A_k-A_{0,k}),$$

where $k$ labels one independent world–driver realization. The two activity axes use different group targets; they do not measure water volumes. Frequencies of joint gains depend on this particular matching of random inputs, rather than providing a general probability of policy success.

The intervals in the saved outcome-comparison table are the original simultaneous intervals for the planned comparisons. Histogram spread is variation between worlds, not a confidence interval for the mean. Sources, agents and steps within one world are dependent observations.


In [ ]:
problems = []
if bundle is not None and (bundle.path / "test_comparisons.json").is_file():
    problems = bundle.problems()

def design_name(pid, record):
    if pid == reference.id:
        return "Original"
    policy = record["policy"]
    observation = {"local": "Local participation", "global": "Population participation", "fixed": "Half participation assumed"}[policy["observation"]]
    forced = f' · forced {policy["forced"]}' if policy.get("forced") else ""
    return f'{observation}: u={policy["u"]:+.4f}, v={policy["v"]:+.4f}{forced} · {pid[:6]}'

def show_distribution(problem=0, design=None):
    records = bundle.designs(problems[problem])
    design = design or next((pid for pid in records if pid != reference.id), None)
    if design is None:
        display(Markdown("This setting contains no changed design to compare."))
        return
    changed, original = bundle.matched(problems[problem], design, reference.id)
    policy = Policy(**changed["policy"])
    observation = {
        "local": "Decisions use previous participation around the selected source, excluding the focal actor.",
        "global": "Decisions use previous participation across the population, excluding the focal actor.",
        "fixed": "Decisions assume half of others restrain, regardless of observed participation.",
    }[policy.observation]
    if policy.forced:
        rule = f"The action is forced to {policy.forced}; the payoff comparison does not choose it."
    elif policy.observation == "fixed":
        a = .5 * (policy.t - 1 - policy.s)
        b = .5 * (policy.u + policy.v)
        rule = f"Restraint is preferred up to {(a+b)/(2*a+b):.1%} stock under that assumption."
    else:
        rule = (f"Restraint is preferred up to {policy.q_d:.1%} stock when others request more, "
                f"and up to {policy.q_c:.1%} when all others restrain.")
    display(Markdown(f"**Selected design:** $u={policy.u:+.4f}$ alongside restraint; $v={policy.v:+.4f}$ despite higher requests. "
                     f"{observation} {rule} The original switches at 50% stock."))
    dh = 100 * (np.asarray(changed["human_services"]) - np.asarray(original["human_services"]))
    da = 100 * (np.asarray(changed["ai_services"]) - np.asarray(original["ai_services"]))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), layout="constrained")
    bins = np.histogram_bin_edges(np.r_[dh, da], bins=24)
    axes[0].hist(dh, bins=bins, alpha=.65, color=BLUE, label="Human")
    axes[0].hist(da, bins=bins, alpha=.6, color=ORANGE, label="AI")
    axes[0].axvline(0, color="black", lw=.7)
    axes[0].set(xlabel="Design − original (percentage points)", ylabel="Matched worlds", title="Changes differ between worlds")
    axes[0].legend(frameon=False)
    axes[1].scatter(dh, da, s=15, alpha=.5, color=TEAL)
    axes[1].axvline(0, color=GRAY, lw=.7); axes[1].axhline(0, color=GRAY, lw=.7)
    axes[1].set(xlabel="Human support change (percentage points)", ylabel="AI support change (percentage points)", title="Upper right: both gain in that world")
    plt.show(); plt.close(fig)
    display(Markdown(f"**{len(dh)} matched worlds.** Both gain in **{((dh>1e-10)&(da>1e-10)).mean():.1%}**; "
                     f"both lose in **{((dh<-1e-10)&(da<-1e-10)).mean():.1%}**. "
                     f"Mean activity changes: human **{dh.mean():+.5f} points**, AI **{da.mean():+.5f} points**."))
    try:
        intervals = bundle.table("paired_outcome_comparisons.csv")
        p = problems[problem]["pressure"]
        rows = intervals[(intervals.pressure == p) & (intervals.policy_id == design)]
        cols = [c for c in ["outcome", "difference", "simultaneous_lower", "simultaneous_upper"] if c in rows]
        display(Markdown("**Original simultaneous intervals, in the saved 0–1 outcome units** (multiply by 100 for percentage points). For linear activity these are target fractions; depletion uses its own bounded score."))
        display(rows[cols].drop_duplicates().reset_index(drop=True))
    except FileNotFoundError:
        display(Markdown("The original interval table is not present in this bundle."))

if problems:
    problem_control = widgets.Dropdown(options=[(f'p={p["pressure"]:g}, B={p["budget"]:g}, {p["norm"]}', i) for i, p in enumerate(problems)], description="Comparison", style={"description_width": "initial"})
    design_control = widgets.Dropdown(description="Design", layout=widgets.Layout(width="580px"))
    def refresh_designs(change=None):
        records = bundle.designs(problems[problem_control.value])
        options = [(design_name(pid, r), pid) for pid, r in records.items() if pid != reference.id]
        design_control.options = options
        design_control.value = options[0][1] if options else None
    refresh_designs()
    problem_control.observe(refresh_designs, names="value")
    distribution_output = widgets.interactive_output(show_distribution, {"problem": problem_control, "design": design_control})
    display(problem_control, design_control, distribution_output)
else:
    display(Markdown("Saved paired test comparisons are not available. The summary-table view above can still be used when its inputs are present."))


## 4 · Follow one matched world

Select a world and a design. Both maps use the **same geometry and colour scale**. A positive stock difference means more water at that location under the selected design, measured in percentage points of that source's capacity. The third map isolates those differences when the two stock maps look similar. Its symmetric colour scale can change between selections; the numeric colour bar gives the magnitude. There is no interpolation between source locations.

The time-series panel compares the same observable under both designs. Shading marks the actual excluded initial window from this bundle's configuration. The vertical line marks the mapped step. Detailed arrays contain post-withdrawal states, including that initial window; the reported outcome means use only their specified observation window.

The cache holds at most four selected histories. Missing detailed files produce an explicit message while the ensemble views remain usable. The saved experiment's robustness comparisons usually contain summaries without full histories; this view uses detailed **test** histories.

Activity curves use the valuation and utility recorded in this bundle. The main study uses linear delivery relative to a fixed target; alternative configured utilities need not equal raw target coverage.


In [ ]:
def show_history(problem=0, design=None, world_index=0, step=1, metric="stock"):
    if design is None:
        return
    changed, original = bundle.matched(problems[problem], design, reference.id)
    if world_index >= len(original["world_ids"]):
        return
    world_id = original["world_ids"][world_index]
    try:
        world, ref = bundle.history(original["realization_ids"][world_index], world_id)
        _, alt = bundle.history(changed["realization_ids"][world_index], world_id)
    except FileNotFoundError:
        display(Markdown("**Detailed files are absent for this selection.** Ensemble comparisons remain available; a world and its corresponding trajectory files are needed for these maps."))
        return
    length = len(ref["stocks"])
    if len(alt["stocks"]) != length:
        raise ValueError("Matching histories have different horizons")
    index = min(step, length) - 1
    difference = 100*(alt["stocks"][index] - ref["stocks"][index])/world.capacity
    fig = plt.figure(figsize=(11.5, 6.5), layout="constrained")
    grid = fig.add_gridspec(2, 3, height_ratios=[1, .7])
    maps = [fig.add_subplot(grid[0, j]) for j in range(3)]
    timeline = fig.add_subplot(grid[1, :])
    im = frame_map(maps[0], world, ref["stocks"][index], ref["strategies"][index], f"Original · step {index+1}")
    frame_map(maps[1], world, alt["stocks"][index], alt["strategies"][index], f"Selected design · step {index+1}")
    fig.colorbar(im, ax=maps[:2], shrink=.7, label="Stock / capacity")
    field = np.full((world.spec.height, world.spec.width), np.nan)
    field[world.resources[:,1], world.resources[:,0]] = difference
    limit = max(.01, float(np.max(np.abs(difference))))
    delta = maps[2].imshow(np.ma.masked_invalid(field), origin="lower", cmap="RdBu_r", vmin=-limit, vmax=limit,
                          extent=(-.5, world.spec.width-.5, -.5, world.spec.height-.5))
    maps[2].set(title="Where stocks changed", xlabel="Grid x", ylabel="Grid y")
    fig.colorbar(delta, ax=maps[2], shrink=.7, label="Stock difference (capacity points)", format="%.2g")
    labels = {"stock": "Mean accessible stock / capacity", "cooperation": "Actors making lower requests", "human": "Human activity measure", "ai": "AI activity measure"}
    for history, label, color in [(ref, "Original", GRAY), (alt, "Selected design", TEAL)]:
        trace = bundle.trace(history, metric, world)
        timeline.plot(np.arange(1, length+1), 100*trace, color=color, label=label, lw=1.2)
    burn = bundle.config["objective"]["excluded_initial_steps"]
    timeline.axvspan(1, burn, color=GRAY, alpha=.12)
    timeline.axvline(index+1, color=ORANGE, lw=1)
    timeline.set(title=labels[metric], xlabel="Recorded step", ylabel="Percent", ylim=(0, 103))
    timeline.legend(frameon=False, fontsize=8, loc="best")
    fig.legend(handles=map_legend(), loc="outside lower center", ncol=4, frameon=False, fontsize=9)
    plt.show(); plt.close(fig)
    display(Markdown(f"World index **{world_index}**, ID `{world_id[:12]}`. At this step, source stock differences range from "
                     f"**{difference.min():+.2f} to {difference.max():+.2f} percentage points of capacity**. "
                     "The request and delivery arrays permit further source- and actor-level analysis."))

if problems:
    world_control = widgets.IntSlider(value=0, min=0, max=0, description="World index", continuous_update=False)
    history_step = widgets.IntSlider(value=min(bundle.config["objective"]["excluded_initial_steps"]+1, bundle.config["objective"]["total_steps"]), min=1, max=bundle.config["objective"]["total_steps"], description="Recorded step", continuous_update=False)
    metric_control = widgets.Dropdown(options=[("Resource stock", "stock"), ("Lower requests", "cooperation"), ("Human support", "human"), ("AI support", "ai")], description="Time series")
    def refresh_worlds(change=None):
        records = bundle.designs(problems[problem_control.value])
        count = len(records[reference.id]["world_ids"])
        world_control.max = count - 1
    refresh_worlds()
    problem_control.observe(refresh_worlds, names="value")
    history_output = widgets.interactive_output(show_history, {"problem": problem_control, "design": design_control,
        "world_index": world_control, "step": history_step, "metric": metric_control})
    display(problem_control, design_control, widgets.HBox([world_control, history_step]), metric_control, history_output)


## 5 · Build another view from the saved arrays

The bundle stores the quantities needed to inspect a physical balance, not just a final score. The following cell checks every source at every recorded step in the **currently selected original-game history**. It reads that single history from the cache and creates no new simulation.

$$z_{j,t}=z_{j,t-1}+R_{j,t}-W_{j,t}.$$

Here $R_{j,t}$ is **actual** renewal after capacity clipping, $W_{j,t}$ is withdrawal, and the first previous stock is the source's full starting capacity. This identity concerns resource accounting; it does not attribute a late activity change to a particular causal pathway.


In [ ]:
if problems:
    records = bundle.designs(problems[problem_control.value])
    original = records[reference.id]
    index = world_control.value
    try:
        world, history = bundle.history(original["realization_ids"][index], original["world_ids"][index])
        previous = np.vstack([world.capacity, history["stocks"][:-1]])
        residual = history["stocks"] - previous - history["resource_renewal"] + history["source_withdrawal"]
        print(f"Maximum absolute source-balance residual: {np.max(np.abs(residual)):.3e} resource units")
        assert np.max(np.abs(residual)) < 1e-8
        print("Available arrays:", ", ".join(history))
    except FileNotFoundError:
        display(Markdown("This balance check becomes available when the selected detailed history is present."))


### Interpreting a new analysis

A new plot can answer a different question from the original search. Its measurement window, independent unit and uncertainty calculation then become part of that analysis. The whole world–driver realization is the independent unit here; pooling its agents or steps as independent replications changes the statistical question.

The selected maps illustrate a world, while the distributions and original simultaneous intervals summarize the tested ensembles. A resource threshold is a stated measurement choice. Low stocks, changed cooperation and a self-maintaining feedback trap are distinct claims.

Further numerical work can use `bundle.table(...)`, `bundle.designs(...)` and `bundle.history(...)` directly. The production saved-data calculations remain in [trajectory_analysis](../src/human_ai_commons/trajectory_analysis/); the standard plots are in [analysis.py](../src/human_ai_commons/analysis.py). The [execution guide](../docs/running-experiments.md) documents complete extraction, plotting from derived data, checksums and exact replay.
